In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import pandas as pd
import numpy as np
import sys
from copy import deepcopy,copy
from datetime import datetime
import pickle
import sys
import matplotlib.pyplot as plt
import warnings
from ipywidgets import IntProgress
from scipy.signal import savgol_filter
from sympy.core.cache import clear_cache
from multiprocessing import Pool
#warnings.filterwarnings("ignore")
#warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore")
stderr_fileno = sys.stderr
sys.stderr = open(os.devnull, 'w')
import gc
def print_class_memory(obj, name="object"):
    from pympler.asizeof import asizeof
    print(f"\nMemory usage per attribute of {name}:")
    for attr, val in vars(obj).items():
        try:
            print(f"  {attr:<20} {asizeof(val)/1024**2:8.2f} MB")
        except Exception:
            pass
import psutil, os
process = psutil.Process(os.getpid())
def print_mem_usage(tag=""):
    mem = process.memory_info().rss / (1024 ** 2)
    print(f"[{tag}] Memory usage: {mem:.2f} MB")

import tracemalloc
import linecache
import os


src = os.path.dirname('/export/home/oriolca/Integral_BMS_Governing_Equations/I-BMS/')
sys.path.append(src)
from mcmc_ode import *
from parallel_ode import *

path = os.path.join(src, "Prior/")
sys.path.append(path)
from fit_prior import read_prior_par

priors = {
    "v2_p3": f"Prior/final_prior_param_sq.named_equations.nv2.np3.2016-09-09 18:49:42.927679.dat",
    "v2_p4": f"Prior/final_prior_param_sq.named_equations.nv2.np4.2016-09-09 18:49:43.056910.dat",
    "v2_p8": f"Prior/final_prior_param_sq.named_equations.nv2.np8.2016-09-09 18:49:42.800618.dat",
}

data=pd.read_csv('../Lotka-Volterra/noise_data/1.0_0.csv')

print(data)

x = data[['x','y']]
t = pd.Series(data[['t']].t)

dt = t.to_numpy()[1] - t.to_numpy()[0]
window_length = 41    # must be odd and <= len(t)
polyorder = 3         # generally 2..5

x_hat = savgol_filter(x['x'], window_length, polyorder, delta=dt, mode='interp')
y_hat = savgol_filter(x['y'], window_length, polyorder, delta=dt, mode='interp')

#x['x'] = x_hat
#x['y'] = y_hat

x_dot = savgol_filter(x['x'], window_length, polyorder, deriv=1, delta=dt, mode='interp')
y_dot = savgol_filter(x['y'], window_length, polyorder, deriv=1, delta=dt, mode='interp')

dx = [pd.DataFrame(data = {'x': x_hat, 'y': y_hat}),pd.DataFrame(data = {'x': x_dot, 'y': y_dot})]

print(x)

print(t)

print(dx)

     Unnamed: 0          x         y     t        dx        dy
0             0  10.759572  4.854358   0.0 -0.057780 -0.934931
1             1  10.545703  4.355556   0.5 -0.083852 -0.799113
2             2  10.296010  1.267922   1.0 -0.096984 -0.671264
3             3  11.017958  5.510302   1.5 -0.087758 -0.555217
4             4  10.403667  2.789376   2.0 -0.052200 -0.454869
..          ...        ...       ...   ...       ...       ...
155         155  18.015525  1.933623  77.5  1.706613 -0.205971
156         156  16.428998  3.309608  78.0  1.987492 -0.472696
157         157  17.467785  0.025648  78.5  2.279009 -0.785711
158         158  21.166165  0.517199  79.0  2.579982 -1.140524
159         159  21.056911  0.092972  79.5  2.889226 -1.532002

[160 rows x 6 columns]
             x         y
0    10.759572  4.854358
1    10.545703  4.355556
2    10.296010  1.267922
3    11.017958  5.510302
4    10.403667  2.789376
..         ...       ...
155  18.015525  1.933623
156  16.428998  3.30

In [ ]:
%%time
XLABS = ['x','y']
n_params = 8

path = os.path.join(src, priors[f"v{len(XLABS)}_p{str(n_params)}"])
prior_par = read_prior_par(path)

Ts=[1] + [1.04**k for k in range(1, 20)]

del prior_par['Nopi_abs']
del prior_par['Nopi2_abs']
del prior_par['Nopi_tan']
del prior_par['Nopi2_tan']
del prior_par['Nopi_sinh']
del prior_par['Nopi2_sinh']
del prior_par['Nopi_cosh']
del prior_par['Nopi2_cosh']
del prior_par['Nopi_tanh']
del prior_par['Nopi2_tanh']


OPS = {
    "sin": 1,
    "cos": 1,
    "exp": 1,
    "pow2": 1,
    "pow3": 1,
    "-": 1,
    "+": 2,
    "*": 2,
    "/": 2,
    "**": 2,
}

def run_chunk(args):
    """
    Run one chunk of 100 MCMC steps in a subprocess.
    Rebuild Numba functions if necessary.
    """
    pms_x, pms_y, mdl, model_mdl_x, model_mdl_y, nsteps = args
    dls = []
    for _ in range(nsteps):
        pms_x.mcmc_step(verbose=False)
        pms_y.mcmc_step(verbose=False)

        pms_x.tree_swap()
        dls.append(pms_x.t1.E.val)
        if pms_x.t1.E.val < mdl:
            mdl = deepcopy(pms_x.t1.E.val)
            model_mdl_x = deepcopy(pms_x)
            model_mdl_y = deepcopy(pms_y)

    return pms_x, pms_y, mdl, model_mdl_x, model_mdl_y,dls


import time
tracemalloc.start()

start = time.time()
pms_x = Parallel(
    ops=OPS,
    Ts=Ts,
    variables=XLABS,
    parameters=["a%d" % i for i in range(n_params)],
    x=x, t=t, dx=dx,
    prior_par=prior_par
)

print('SIZE OF PMS_X', print_mem_usage())
print_class_memory(pms_x)
print_class_memory(pms_x.t1)

pms_y = Parallel(
    ops=OPS,
    Ts=Ts,
    variables=XLABS,
    parameters=["a%d" % i for i in range(n_params)],
    x=x, t=t, dx=dx,
    prior_par=prior_par
)

couplings=[pms_x,pms_y]
pms_x.set_couplings(couplings)
snap1 = tracemalloc.take_snapshot()


mdl = np.inf
dl=[]

last_seen_by_can, last_seen_by_str = {}, {}
print(pms_x.t1.fit_par)
for coup in pms_x.trees.values():
    last_seen_by_can[tuple(f.canonical() for f in coup.couplings)] = 0
    last_seen_by_str[tuple(str(f) for f in coup.couplings)] = 0
NCLEAN = 2

# Main loop
nchunks = 80
CHUNK_SIZE = 50
f = IntProgress(min=0, max=nchunks, description='Running:') # instantiate the bar
display(f)
# Initial models
mdl = copy(pms_x.t1.E.val)
model_x = deepcopy(pms_x)
model_y = deepcopy(pms_y)

for i in range(nchunks):
    args = [(pms_x, pms_y, mdl, model_x, model_y, CHUNK_SIZE)]
    with Pool(processes=1, maxtasksperchild=1) as pool:
        for result in pool.imap(run_chunk, args):
            pms_x, pms_y, mdl, model_mdl_x, model_mdl_y,chunck_dl = result
    print(f"Chunk {i+1}/{nchunks} done, last best mdl: {mdl}")
    dl += chunck_dl
    f.value+=1
    """
    for coup in pms_x.trees.values():
        last_seen_by_can[tuple(f.canonical() for f in coup.couplings)] = k
        last_seen_by_str[tuple(str(f) for f in coup.couplings)] = k
    if (k % NCLEAN) == 0:
        to_remove = []
        for represent in pms_x.t1.representative:
            try:
                if last_seen_by_can[represent] < (k - NCLEAN):
                    to_remove.append(represent)
            except KeyError:  # This tree was tested but not visited anyway!
                to_remove.append(represent)
        for t in to_remove:
            del pms_x.t1.representative[t]
            if t in last_seen_by_can:
                del last_seen_by_can[t]
        to_remove = []
        for string in pms_x.t1.fit_par:
            try:
                if last_seen_by_str[string] < (k - NCLEAN):
                    to_remove.append(string)
            except KeyError:  # This tree was tested but not visited anyway!
                to_remove.append(string)
        for t in to_remove:
            del pms_x.t1.fit_par[t]
            del pms_x.t1.fit_x0[t]
            del pms_x.t1.fit_sse[t]
            if t in last_seen_by_str:
                del last_seen_by_str[t]
    """
    
    print('After chunk cache')
    #if k%5==0:
    print(f'{i}--{pms_x.t1}({pms_x.t1.E.val})|{pms_y.t1}({pms_y.t1.E.val})',end='\r')
    #if k%50==0:
    clear_cache()
    print()
    print('SIZE OF PMS_X',print_mem_usage())
    """
    print_class_memory(pms_x)
    print_class_memory(pms_y)
    print_class_memory(pms_x.t1)
    gc.collect()
    snapshot = tracemalloc.take_snapshot()
    top_stats = snapshot.statistics('lineno')

    print(f"\nTop memory allocations at step {i}")
    for stat in top_stats[:15]:
        frame = stat.traceback[0]
        print(f"{os.path.basename(frame.filename)}:{frame.lineno}: "
              f"{stat.size/1024**2:.2f} MB — {linecache.getline(frame.filename, frame.lineno).strip()}")
    diff = snapshot.compare_to(snap1, 'filename')
    print("\nTop memory growth:")
    for stat in diff:
        print(stat)
    """

print('PT (T=1) final model',pms_x.t1.E.val,pms_x.t1)
print('PT (T=1) final model',pms_y.t1.E.val,pms_y.t1)

print(model_x)
print(model_x.E.val)
print(model_x.par_values)
print(model_x.x0_values)

plt.plot(dl)
plt.show()

[] Memory usage: 552.55 MB
SIZE OF PMS_X None

Memory usage per attribute of object:
  Ts                       0.00 MB
  trees                    1.86 MB
  t1                       0.16 MB
  couplings                0.00 MB

Memory usage per attribute of object:
  variables                0.00 MB
  parameters               0.00 MB
  couplings                0.00 MB
  root                     0.00 MB
  ops                      0.00 MB
  op_orders                0.00 MB
  move_types               0.00 MB
  ets                      0.00 MB
  dist_par                 0.00 MB
  n_dist_par               0.00 MB
  nodes                    0.00 MB
  size                     0.00 MB
  max_size                 0.00 MB
  et_space                 0.08 MB
  rr_space                 0.01 MB
  num_rr                   0.00 MB
  nops                     0.00 MB
  prior_par                0.00 MB
  x                        0.04 MB
  dx                       0.02 MB
  t                        0.00 MB
 

IntProgress(value=0, description='Running:', max=80)

Chunk 1/80 done, last best mdl: 595.071843441182
After chunk cache
0--(_a0_ * _a0_)(599.7731274541989)|_a0_(599.7731274541989)
[] Memory usage: 731.54 MB
SIZE OF PMS_X None
Chunk 2/80 done, last best mdl: 595.071843441182
After chunk cache
1--_a5_(595.071843441182)|_a0_(595.071843441182)
[] Memory usage: 775.66 MB
SIZE OF PMS_X None
Chunk 3/80 done, last best mdl: 595.071843441182
After chunk cache
2--_a5_(595.4381869165077)|_a5_(595.4381869165077)
[] Memory usage: 819.54 MB
SIZE OF PMS_X None
Chunk 4/80 done, last best mdl: 595.071843441182
After chunk cache
3--_a5_(595.4381869165077)|_a5_(595.4381869165077)
[] Memory usage: 850.66 MB
SIZE OF PMS_X None
Chunk 5/80 done, last best mdl: 595.071843441182
After chunk cache
4--_a5_(595.4381869165077)|_a5_(595.4381869165077)
[] Memory usage: 867.54 MB
SIZE OF PMS_X None
Chunk 6/80 done, last best mdl: 595.071843441182
After chunk cache
5--_a5_(595.4381869165077)|_a5_(595.4381869165077)
[] Memory usage: 869.54 MB
SIZE OF PMS_X None
Chunk 7/8

In [ ]:
integral = model_x.integrate(t)

plt.plot(t,integral['d0']['x'])
plt.plot(t,x['x'])
plt.plot(t,integral['d0']['y'])
plt.plot(t,x['y'])

In [ ]:
dx_pred = model_x.predict(x)

plt.plot(x['x'],dx_pred['x'])
plt.plot(x['x'],dx['x'])
plt.plot(x['y'],dx_pred['y'])
plt.plot(x['y'],dx['y'])